In [3]:
import pandas as pd
import sqlite3
 
conn = sqlite3.connect('lab10.db')
 
df = pd.read_csv('Lab_10_data(Fact sales).csv', sep=';')
df.to_sql('fact_sales', conn, if_exists='replace', index=False)
print("✓ fact_sales загружена")
print(df)
print()

✓ fact_sales загружена
   sales_key  order_id  product_key  customer_key  date_key  quantity  \
0          1      1001            1             1  20240115         2   
1          2      1001            2             1  20240115         1   
2          3      1002            4             2  20240210         1   
3          4      1003            3             3  20240305         1   
4          5      1003            5             3  20240305         3   
5          6      1004            6             4  20240305         1   
6          7      1004            1             4  20240305         1   
7          8      1005            2             2  20240401         2   
8          9      1006            3             1  20240520         1   
9         10      1006            5             1  20240520         2   

   unit_price  revenue  
0       49.99    99.98  
1       39.99    39.99  
2       14.99    14.99  
3       45.00    45.00  
4       10.99    32.97  
5       12.99    12.99 

In [4]:
df_product = pd.DataFrame([
    (1, 'P001', 'The Great Gatsby',    'F. Scott Fitzgerald', 'Classic',     49.99),
    (2, 'P002', 'Dune',                'Frank Herbert',       'Sci-Fi',      39.99),
    (3, 'P003', 'Atomic Habits',       'James Clear',         'Self-Help',   45.00),
    (4, 'P004', 'The Alchemist',       'Paulo Coelho',        'Fiction',     14.99),
    (5, 'P005', 'Python Crash Course', 'Eric Matthes',        'Technology',  10.99),
    (6, 'P006', 'Sapiens',             'Yuval Noah Harari',   'Non-Fiction', 12.99),
], columns=['product_key', 'product_id', 'title', 'author', 'genre', 'price'])
df_product.to_sql('dim_product', conn, if_exists='replace', index=False)
 
df_customer = pd.DataFrame([
    (1, 'C1001', 'Alice Johnson', 'alice@email.com', 'Gold',   '2022-01-15'),
    (2, 'C1002', 'Bob Smith',     'bob@email.com',   'Silver', '2021-06-20'),
    (3, 'C1003', 'Carol Davis',   'carol@email.com', 'Bronze', '2023-03-10'),
    (4, 'C1004', 'David Lee',     'david@email.com', 'Silver', '2020-11-05'),
], columns=['customer_key', 'customer_id', 'name', 'email', 'loyalty_tier', 'registration_date'])
df_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
 
df_date = pd.DataFrame([
    (20240115, '2024-01-15', 2024, 1, 1, 'January',  'Monday'),
    (20240210, '2024-02-10', 2024, 1, 2, 'February', 'Saturday'),
    (20240305, '2024-03-05', 2024, 1, 3, 'March',    'Tuesday'),
    (20240401, '2024-04-01', 2024, 2, 4, 'April',    'Monday'),
    (20240520, '2024-05-20', 2024, 2, 5, 'May',      'Monday'),
], columns=['date_key', 'date', 'year', 'quarter', 'month', 'month_name', 'day_of_week'])
df_date.to_sql('dim_date', conn, if_exists='replace', index=False)
print("✓ Все измерения загружены")
print()

✓ Все измерения загружены



In [5]:
def query(title, sql):
    print("=" * 55)
    print(title)
    print("=" * 55)
    print(pd.read_sql(sql, conn).to_string(index=False))
    print()

In [6]:
 
query("Query 2 — Orders and revenue per loyalty tier", """
    SELECT c.loyalty_tier,
           COUNT(DISTINCT f.order_id) AS num_orders,
           ROUND(SUM(f.revenue), 2) AS total_revenue
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.loyalty_tier ORDER BY total_revenue DESC
""")

Query 2 — Orders and revenue per loyalty tier
loyalty_tier  num_orders  total_revenue
        Gold           2         206.95
      Silver           3         157.95
      Bronze           1          77.97



In [7]:
query("Query 2 — Orders and revenue per loyalty tier", """
    SELECT c.loyalty_tier,
           COUNT(DISTINCT f.order_id) AS num_orders,
           ROUND(SUM(f.revenue), 2) AS total_revenue
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.loyalty_tier ORDER BY total_revenue DESC
""")

Query 2 — Orders and revenue per loyalty tier
loyalty_tier  num_orders  total_revenue
        Gold           2         206.95
      Silver           3         157.95
      Bronze           1          77.97



In [8]:
query("Query 3 — Monthly total revenue for 2024", """
    SELECT d.month_name, ROUND(SUM(f.revenue), 2) AS monthly_revenue
    FROM fact_sales f
    JOIN dim_date d ON f.date_key = d.date_key
    WHERE d.year = 2024
    GROUP BY d.month, d.month_name ORDER BY d.month
""")

Query 3 — Monthly total revenue for 2024
month_name  monthly_revenue
   January           139.97
  February            14.99
     March           140.95
     April            79.98
       May            66.98



In [9]:
query("Query 4 — Top 3 best-selling products", """
    SELECT p.title, SUM(f.quantity) AS total_qty
    FROM fact_sales f
    JOIN dim_product p ON f.product_key = p.product_key
    GROUP BY p.title ORDER BY total_qty DESC LIMIT 3
""")
 

Query 4 — Top 3 best-selling products
              title  total_qty
Python Crash Course          5
   The Great Gatsby          3
               Dune          3



In [10]:
query("Query 5 — Average order value", """
    SELECT ROUND(SUM(revenue) / COUNT(DISTINCT order_id), 2) AS avg_order_value
    FROM fact_sales
""")

Query 5 — Average order value
 avg_order_value
           73.81



In [11]:
query("Query 6 — Customers who spent more than $100", """
    SELECT c.name, ROUND(SUM(f.revenue), 2) AS total_spent
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.name HAVING total_spent > 100 ORDER BY total_spent DESC
""")
 
print("=" * 55)
print("Part 3 — SCD Type 2")
print("=" * 55)

Query 6 — Customers who spent more than $100
         name  total_spent
Alice Johnson       206.95

Part 3 — SCD Type 2


In [12]:
df_scd2 = pd.DataFrame([
    (1, 'C1001', 'Alice Johnson', 'alice@email.com', 'Gold',   '2022-01-15', '2022-01-15', None, 1),
    (2, 'C1002', 'Bob Smith',     'bob@email.com',   'Silver', '2021-06-20', '2021-06-20', None, 1),
    (3, 'C1003', 'Carol Davis',   'carol@email.com', 'Bronze', '2023-03-10', '2023-03-10', None, 1),
    (4, 'C1004', 'David Lee',     'david@email.com', 'Silver', '2020-11-05', '2020-11-05', None, 1),
], columns=['customer_key','customer_id','name','email',
            'loyalty_tier','registration_date','effective_date','expiry_date','is_current'])
df_scd2.to_sql('dim_customer_scd2', conn, if_exists='replace', index=False)
 
conn.execute("""
    UPDATE dim_customer_scd2
    SET expiry_date = '2024-03-31', is_current = 0
    WHERE customer_id = 'C1001' AND is_current = 1
""")
 
new_row = pd.DataFrame([{
    'customer_key': 101, 'customer_id': 'C1001',
    'name': 'Alice Johnson', 'email': 'alice@email.com',
    'loyalty_tier': 'Platinum', 'registration_date': '2022-01-15',
    'effective_date': '2024-04-01', 'expiry_date': None, 'is_current': 1
}])
new_row.to_sql('dim_customer_scd2', conn, if_exists='append', index=False)
conn.commit()
 
print("История Alice Johnson:")
print(pd.read_sql("""
    SELECT customer_key, loyalty_tier, effective_date, expiry_date, is_current
    FROM dim_customer_scd2 WHERE customer_id = 'C1001' ORDER BY effective_date
""", conn).to_string(index=False))
 
conn.close()
print()
print("✓ Готово! База данных сохранена в lab10.db")

История Alice Johnson:
 customer_key loyalty_tier effective_date expiry_date  is_current
            1         Gold     2022-01-15  2024-03-31           0
          101     Platinum     2024-04-01        None           1

✓ Готово! База данных сохранена в lab10.db
